# todo
1. scene/gaussian_model.py
    - restore(given)
2. gaussian_renderer/__init__.py (given)
3. cuda code
    - forward.cu

# arguments/__init__.py


In [ ]:
class PipelineParams(ParamGroup):
    def __init__(self, parser):
        self.convert_SHs_python = False
        self.compute_cov3D_python = False
        self.debug = False
        self.env_map_res = 0
        self.env_optimize_until = 1000000000
        self.env_optimize_from = 0
        self.eval_shfs_4d = False

        # ========= added line start here =========
        self.opa_threshold = 0.05
        # ========= added line end here =========
        
        super().__init__(parser, "Pipeline Parameters")

# gaussian_renderer/diff_gaussian_rasterization.py

In [ ]:
@staticmethod
def forward(
     ctx,
     means3D,
     means2D,
     sh,
     colors_precomp,
     flow_2d,
     opacities,
     ts,
     scales,
     scales_t,
     rotations,
     rotations_r,
     cov3Ds_precomp,
     raster_settings,
):
     args = (
          raster_settings.bg,
          means3D,
          colors_precomp,
          flow_2d,
          opacities,
          ts,
          scales,
          scales_t,
          rotations,
          rotations_r,
          raster_settings.scale_modifier,
          cov3Ds_precomp,
          # ────────── static Gaussian parameters (forward) ──────────
          raster_settings.static_xyz,
          raster_settings.static_features_dc,
          raster_settings.static_features_rest,
          raster_settings.static_scaling,
          raster_settings.static_rotation,
          raster_settings.static_opacity,
          raster_settings.static_max_radii2D,
          raster_settings.static_denom,
          raster_settings.static_xyz_gradient_accum,
          # ────────────────────────────────────────────────────────────
          raster_settings.viewmatrix,
          raster_settings.projmatrix,
          raster_settings.tanfovx,
          raster_settings.tanfovy,
          raster_settings.image_height,
          raster_settings.image_width,
          sh,
          raster_settings.sh_degree,
          raster_settings.sh_degree_t,
          raster_settings.campos,
          raster_settings.timestamp,
          raster_settings.time_duration,
          raster_settings.rot_4d,
          raster_settings.gaussian_dim,
          raster_settings.force_sh_3d,
          raster_settings.prefiltered,
          raster_settings.debug,
     )
#    …
@staticmethod
def backward(ctx, grad_out_color, grad_radii, grad_depth, grad_alpha, grad_flow, grad_covs_com):
     raster_settings = ctx.raster_settings
#    …
     args = (
          raster_settings.bg,
          means3D,
          out_means3D,
          radii,
          colors_precomp,
          flow_2d,
          opacities,
          ts,
          scales,
          scales_t,
          rotations,
          rotations_r,
          raster_settings.scale_modifier,
          cov3Ds_precomp,
          # ────────── static Gaussian parameters (backward) ─────────
          raster_settings.static_xyz,
          raster_settings.static_features_dc,
          raster_settings.static_features_rest,
          raster_settings.static_scaling,
          raster_settings.static_rotation,
          raster_settings.static_opacity,
          raster_settings.static_max_radii2D,
          raster_settings.static_denom,
          raster_settings.static_xyz_gradient_accum,
          # ────────────────────────────────────────────────────────────
          raster_settings.viewmatrix,
          raster_settings.projmatrix,
          raster_settings.tanfovx,
          raster_settings.tanfovy,
          grad_out_color,
          grad_depth,
          grad_alpha,
          grad_flow,
          sh,
          raster_settings.sh_degree,
          raster_settings.sh_degree_t,
          raster_settings.campos,
          raster_settings.timestamp,
          raster_settings.time_duration,
          raster_settings.rot_4d,
          raster_settings.gaussian_dim,
          raster_settings.force_sh_3d,
          geomBuffer,
          num_rendered,
          binningBuffer,
          imgBuffer,
          raster_settings.debug
     )
#    …


# scene/gaussian_model.py

In [ ]:
def __init__(self, sh_degree: int, gaussian_dim: int = 3, …):
    …
    self.setup_functions()

    # --- Hybrid 3D–4D용 static Gaussian placeholder -------------
    # restore() 시 unpack 될 필드들이 미리 있어야 합니다.
    self.static_xyz                = torch.empty(0)              # [N_static × 3]
    self.static_features_dc        = torch.empty(0)              # [N_static × DC]
    self.static_features_rest      = torch.empty(0)              # [N_static × Rest]
    self.static_scaling            = torch.empty(0)              # [N_static × 3]
    self.static_rotation           = torch.empty(0)              # [N_static × 4]
    self.static_opacity            = torch.empty(0)              # [N_static × 1]
    self.static_max_radii2D        = torch.empty(0, dtype=torch.int64)
    self.static_denom              = torch.empty(0)              # [N_static × 1]
    self.static_xyz_gradient_accum = torch.empty(0)              # [N_static × 1]

def capture(self):
    if self.gaussian_dim == 3:
        return ( … )       # (기존 3D 리턴 그대로)
    elif self.gaussian_dim == 4:
        return (
            self.active_sh_degree,
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
            self.max_radii2D,
            self.xyz_gradient_accum,
            self.t_gradient_accum,
            self.denom,
            self.optimizer.state_dict(),
            self.spatial_lr_scale,
            self._t,
            self._scaling_t,
            self._rotation_r,
            self.rot_4d,
            self.env_map,
            self.active_sh_degree_t,
            # ────────── 여기부터 새로운 static 필드 ──────────
            self.static_xyz,
            self.static_features_dc,
            self.static_features_rest,
            self.static_scaling,
            self.static_rotation,
            self.static_opacity,
            self.static_max_radii2D,
            self.static_denom,
            self.static_xyz_gradient_accum,
        )

# cuda_rasterizer/forward.h 

In [ ]:
@@ // 기존
- void preprocess(int P, int D, int D_t, int M,
+ void preprocess(int P, int D, int D_t, int M,
                const float* orig_points,
                float* out_means3D,
+               // ─── 여기에 static Gaussian args ───
+               const float* static_xyz,
+               const float* static_features_dc,
+               const float* static_features_rest,
+               const glm::vec3* static_scaling,
+               const glm::vec4* static_rotation,
+               const float* static_opacity,
+               const int32_t* static_max_radii2D,
+               const float* static_denom,
+               const float* static_xyz_grad_accum,
+               // ───────────────────────────────────
                const float* ts,
                /* … 나머지 인자 그대로 … */
                bool prefiltered);


In [ ]:
template<int C>
__global__ void preprocessCUDA(
    /* ... 기존 인자들 ... */,
    const float* static_xyz,
    const glm::vec3* static_scaling,
    const glm::vec4* static_rotation,
    const float* static_opacity,
    const float* static_features_dc,
    const float* static_features_rest,
    const int*   static_max_radii2D,
    const float* static_denom,
    const float* static_xyz_grad_accum,
    /* ... 나머지 인자들 ... */
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= P) return;

    // 1) 원본 포인트 & opacity
    float3 p_orig = { orig_points[3*idx+0], orig_points[3*idx+1], orig_points[3*idx+2] };
    float  op      = opacities[idx];

    bool is_dynamic = false;
    float3 cov3D_storage[6];
    const float* cov3D_ptr = nullptr;

    // 2) dynamic(4D) 경로 시도
    if (rot_4d) {
        bool time_mask = true;
        computeCov3D_conditional(
            scales[idx], scales_t[idx], scale_modifier,
            rotations[idx], rotations_r[idx],
            cov3D_storage,    // 임시 저장소
            p_orig, ts[idx], timestamp, idx,
            time_mask, op
        );
        if (time_mask) {
            // → dynamic 으로 분류됨
            is_dynamic = true;
            cov3D_ptr = cov3D_storage;
        }
    }

    // 3) static(3D) 경로
    if (!is_dynamic) {
        // 3-1) static 위치 덮어쓰기
        float sx = static_xyz[3*idx+0],
              sy = static_xyz[3*idx+1],
              sz = static_xyz[3*idx+2];
        p_orig = { sx, sy, sz };

        // 3-2) static용 3D 공분산 계산
        computeCov3D(
            static_scaling[idx],
            scale_modifier,
            static_rotation[idx],
            cov3D_storage
        );
        cov3D_ptr = cov3D_storage;

        // 3-3) opacity, SH feature 등도 static 값으로 교체
        op      = static_opacity[idx];
        shs     = static_features_dc + idx * maxSH;   // DC 차원만큼
        shs_rest = static_features_rest + idx * maxRest;
        max_r  = static_max_radii2D[idx];
        denom  = static_denom[idx];
        // 필요하면 xyz_grad_accum도 p_orig 에 더해줄 수 있음
    }

    // 이제 cov3D_ptr, p_orig, op, shs 등은
    // dynamic·static 구분 없이 “통합된” 리스트로 후처리됩니다.

    // 4) 프러스텀 컬링
    if (!in_frustum(p_orig, viewmatrix, projmatrix, prefiltered)) return;

    // 5) 투영 → 2D 공분산 → 반지름 계산 → 타일 지정
    float4 p_hom = transformPoint4x4(p_orig, projmatrix);
    // … (이하 기존 코드와 동일) …

    // 6) 저장
    out_means3D[3*idx+0] = p_orig.x;
    out_means3D[3*idx+1] = p_orig.y;
    out_means3D[3*idx+2] = p_orig.z;
    cov3Ds[6*idx + 0..5] = *cov3D_ptr;
    opacities_out[idx]   = op;
    // …
}


In [ ]:
// 1) precomputed cov 체크
if (cov3D_precomp) { … }

// 2) dynamic(4D) 경로
bool is_dynamic = false;
if (rot_4d) {
  bool time_mask = true;
  computeCov3D_conditional(..., time_mask, opacity);
  if (time_mask) {
    is_dynamic = true;
    // cov3D_ptr, p_orig, opacity 이미 셋업됨
  }
}

// 3) static(3D) 경로
if (!is_dynamic) {
  // 위치, 공분산, opacity, SH, max_radii2D, denom 등
  // 모두 static_* 배열에서 꺼내서 덮어쓰기
  p_orig          = { static_xyz[3*i], … };
  computeCov3D(static_scaling[i], …, cov3D_storage);
  opacity         = static_opacity[i];
  shs             = static_features_dc + i*DC;
  shs_rest        = static_features_rest + i*Rest;
  max_radii2D     = static_max_radii2D[i];
  denom           = static_denom[i];
  // (xyz_gradient_accum 필요시 더해주기)
  cov3D_ptr       = cov3D_storage;
}

// 4) 공통 후처리: in_frustum → computeCov2D → radii 계산 → 타일 범위 지정 → tile 리스트에 append


In [ ]:
bool is_dynamic = false;
float3 p = /* orig_points[idx] */;
float   op = opacities[idx];
float   cov3D_storage[6];

// 1) 4D 조건부 경로
if (rot_4d) {
    bool time_mask = true;
    computeCov3D_conditional(/*…*/, cov3D_storage, p, /*…*/, time_mask, op);
    if (time_mask) {
        is_dynamic = true;
        cov3D = cov3D_storage;
        out_means3D[3*idx + ...] = p.x, p.y, p.z;
    }
}

// 2) static(3D) 경로
if (!is_dynamic) {
    // 위치 교체
    p.x = static_xyz[3*idx+0];
    p.y = static_xyz[3*idx+1];
    p.z = static_xyz[3*idx+2];
    // 3D 공분산
    computeCov3D(static_scaling[idx], scale_modifier, static_rotation[idx], cov3D_storage);
    cov3D = cov3D_storage;
    // 불투명도·SH·max_radii·denom 덮어쓰기
    op = static_opacity[idx];
    shs       = static_features_dc    + idx*DC;
    shs_rest  = static_features_rest  + idx*Rest;
    max_r     = static_max_radii2D[idx];
    denom     = static_denom[idx];
    // (원하면 static_xyz_grad_accum를 p에 더해줄 수도)
    out_means3D[3*idx ...] = p.x, p.y, p.z;
}

// 3) (공통) 프러스텀 컬링 → 2D cov → tile 범위 계산 → 
//    radii, points_xy_image, depths, conic_opacity 저장 → tiles_touched


In [ ]:
// 커널 상단
bool is_dynamic = false;
float3 cov3D_storage[6];
const float* cov3D = nullptr;
glm::vec3 p_orig = /* orig_points[idx] */;
float opacity    = /* opacities[idx] */;

// ① dynamic(4D) 경로
if (rot_4d) {
  bool time_mask = true;
  computeCov3D_conditional(
    scales[idx], scales_t[idx], scale_modifier,
    rotations[idx], rotations_r[idx],
    cov3D_storage, p_orig, ts[idx], timestamp,
    idx, time_mask, opacity
  );
  if (time_mask) {
    is_dynamic = true;
    cov3D = cov3D_storage;
    out_means3D[idx*3+0] = p_orig.x;
    out_means3D[idx*3+1] = p_orig.y;
    out_means3D[idx*3+2] = p_orig.z;
  }
}

// ② static(3D) 경로
if (!is_dynamic) {
  // 위치 덮어쓰기
  p_orig.x = static_xyz[3*idx+0];
  p_orig.y = static_xyz[3*idx+1];
  p_orig.z = static_xyz[3*idx+2];
  // 3D-only 공분산
  computeCov3D(
    static_scaling[idx],
    scale_modifier,
    static_rotation[idx],
    cov3D_storage
  );
  cov3D = cov3D_storage;
  // 불투명도·SH·radii·denom 덮어쓰기
  opacity = static_opacity[idx];
  shs     = static_features_dc   + idx * DC;
  shs_rest= static_features_rest + idx * Rest;
  max_r   = static_max_radii2D[idx];
  denom   = static_denom[idx];
  // 원하면 static_xyz_grad_accum 을 p_orig 에 더해주기
  out_means3D[idx*3+0] = p_orig.x;
  out_means3D[idx*3+1] = p_orig.y;
  out_means3D[idx*3+2] = p_orig.z;
}

// ③ 공통 후처리: frustum culling → computeCov2D → tile 리스트 작성
